# Práctica de Procesamiento de Lenguaje Natural (PLN)
## Preparación de corpus, vocabularios y entrenamiento de un modelo neuronal de lenguaje

---

### Contenidos de la práctica

Construiremos **desde cero** un modelo neuronal capaz de generar texto en español a partir de letras de canciones. El proceso completo tiene 7 pasos:

| Paso | Descripción |
|------|-------------|
| 1 | **Lectura y tokenización** del corpus |
| 2 | **Construcción del vocabulario** (palabra → tokenID) |
| 3 | **Preparación del dataset** (pares contexto → siguiente palabra) |
| 4 | **Definición del modelo** neuronal (arquitectura de Bengio, 2003) |
| 5 | **Dataset y DataLoader** de PyTorch |
| 6 | **Entrenamiento** del modelo |
| 7 | **Generación de texto** con muestreo Top-K |

### El modelo: Red Neuronal de Bengio
Propuesto por Yoshua Bengio en 2003, fue uno de los primeros modelos de lenguaje basados en redes neuronales. La idea central es: **dadas las N palabras anteriores, predice cuál es la siguiente**.

---

In [2]:
# En Google Colab PyTorch ya está disponible.
# Si trabajas en local y no lo tienes, descomenta la siguiente línea:
# !pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import re
from tqdm import tqdm

print(f'PyTorch versión: {torch.__version__}')
print(f'GPU disponible:  {torch.cuda.is_available()}')
print('OK - Librerías importadas correctamente.')

PyTorch versión: 2.11.0+cu130
GPU disponible:  True
OK - Librerías importadas correctamente.


---
## PASO 1: Lectura y Tokenización del Corpus

### Corpus para entrenamiento
Un **corpus** es una colección de textos que usamos para entrenar el modelo. En esta práctica son letras de canciones.

### Formato del archivo
Cada canción está delimitada por las etiquetas `<CS>` y `</CS>`. Dentro de cada canción, cada verso termina con el token especial `<EOL>` (End Of Line):

```
<CS>
Ya te dije frases hechas y deshechas <EOL>
Ya he soñado algún momento de tu piel <EOL>
</CS>
<CS>
Vida de mi vida <EOL>
¿Adónde hemos llegado? <EOL>
</CS>
```

### ¿Qué es tokenizar?
**Tokenizar** es dividir el texto en unidades mínimas llamadas **tokens**. Aquí usamos las palabras como tokens, separando por espacios. El token `<EOL>` ya viene marcado en el corpus: el modelo aprenderá qué palabras suelen terminar un verso.

### Resultado esperado
```python
[
  ['Ya', 'te', 'dije', 'frases', 'hechas', 'y', 'deshechas', '<EOL>', ...],
  ['Vida', 'de', 'mi', 'vida', '<EOL>', ...],
  ...
]
```

In [3]:
import re

def read_and_tokenize(filename):
    """
    Lee un corpus de canciones con formato <CS>...</CS> y lo tokeniza.

    Tokens especiales preservados: <CS>, </CS>, <EOL>
    El resto de puntuación se separa en tokens individuales.
    """
    with open(filename, 'r', encoding='utf-8-sig') as f:
        content = f.read()

    # Patrón que captura: <CS>, </CS>, <EOL>, palabras, y puntuación individual
    tokens = re.findall(r'<CS>|</CS>|<EOL>|\w+|[^\w\s<>/]', content)

    # Agrupamos por canciones: iniciamos nueva lista cada vez que vemos <CS>
    sentences = []
    current_song = []

    for token in tokens:
        if token == '<CS>':
            # Si hay canción en curso, la guardamos (por si hay CS anidados o sueltos)
            if current_song:
                sentences.append(current_song)
            current_song = []
        elif token == '</CS>':
            if current_song:
                sentences.append(current_song)
            current_song = []
        else:
            current_song.append(token)

    # Por si queda algo sin cerrar
    if current_song:
        sentences.append(current_song)

    return sentences

# Carga del corpus
Actualiza la variable path_corpus con tu corpus de trabajo

In [4]:
path_corpus = 'corpusA.txt'
sentences = read_and_tokenize(path_corpus)

print(f'Lectura y tokenización del corpus finalizada.')
print(f"🔢 Total de sentencias en el corpus: {len(sentences)}")


sentences = sentences[:1000]  # Primeras 1000 canciones para agilizar


# -----------------------------------------------------------------------
print(f'Total de canciones cargadas: {len(sentences)}')
print(f'\nPrimera cancion ({len(sentences[0])} tokens):')
print(sentences[0])
print(f'\nTotal de tokens en el corpus: {sum(len(s) for s in sentences)}')

Lectura y tokenización del corpus finalizada.
🔢 Total de sentencias en el corpus: 6606
Total de canciones cargadas: 1000

Primera cancion (184 tokens):
['Tú', 'y', 'yo', 'en', 'mi', 'habitación', '<EOL>', 'Y', 'el', 'mundo', 'fuera', '<EOL>', 'Puede', 'que', 'parezca', 'una', 'noche', 'más', '<EOL>', 'Pero', 'yo', 'sé', 'que', 'así', 'esparzo', 'amor', '<EOL>', 'Y', 'mañana', 'cuando', 'salga', 'el', 'sol', '<EOL>', 'Nos', 'querremos', 'aún', 'más', '<EOL>', 'Y', 'al', 'fin', 'te', 'podré', 'abrazar', '<EOL>', 'Como', 'lo', 'hacíamos', 'antes', '<EOL>', 'Yo', 'me', 'siento', 'solo', 'por', 'sólo', 'un', 'instante', '<EOL>', 'Pero', 'sé', 'que', 'cuando', 'pase', 'todo', 'esto', '<EOL>', 'Tú', 'y', 'yo', 'volveremos', 'a', 'ser', 'como', 'éramos', 'antes', '<EOL>', 'Y', 'cuando', 'ya', 'esté', 'seguro', 'que', 'no', 'pueda', 'herir', 'a', 'nadie', '<EOL>', 'Entonces', 'mis', 'pasos', 'me', 'llevarán', 'fuera', 'de', 'este', 'caso', '<EOL>', 'Me', 'acercarán', 'hasta', 'donde', 'tú', 'es

---
## PASO 2: Construcción del Vocabulario

### ¿Por qué necesitamos un vocabulario?
Las redes neuronales operan con **números**, no con texto. El vocabulario es un **diccionario bidireccional** que permite transformar el texto en números:
- `"vida"` → `5`  (tokenID para alimentar la red neuronal)
- `5` → `"vida"` (para interpretar la salida y generar texto legible)

### Tokens especiales
Reservamos dos entradas especiales **desde el principio**:

- **`<PAD>`** *(índice 0)*: Relleno para el inicio de cada canción, donde todavía no hay palabras previas (lo veremos en el Paso 3).


### Tamaño del vocabulario
Una entrada por cada **palabra única** del corpus, más los tokens especiales. Si el corpus tiene 50.000 palabras distintas, el vocabulario tendrá 50.002 entradas.

In [5]:
class Vocabulary:
    """
    Diccionario bidireccional: palabra <-> indice numerico.

    Atributos:
        token_to_idx (dict): 'vida' -> 5
        idx_to_token (dict):  5 -> 'vida'
        size (int): numero total de tokens registrados
    """

    def __init__(self):
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.size = 0

    def add_token(self, token):
        """Añade un token al vocabulario si no existe ya."""
        if token not in self.token_to_idx:
            self.token_to_idx[token] = self.size
            self.idx_to_token[self.size] = token
            self.size += 1

    def encode(self, tokens):
        """
        Convierte lista de palabras en lista de indices numericos.
        Las palabras desconocidas se mapean al indice de <UNK>.
        Ejemplo: ['vida', 'de', 'mi'] -> [5, 8, 11]
        """
        unk_idx = self.token_to_idx.get('<UNK>', 0)
        return [self.token_to_idx.get(tok, unk_idx) for tok in tokens]

    def decode(self, indices):
        """
        Convierte lista de indices en lista de palabras.
        Ejemplo: [5, 8, 11] -> ['vida', 'de', 'mi']
        """
        return [self.idx_to_token[idx] for idx in indices]


# Construimos el vocabulario
vocab = Vocabulary()

# Los tokens especiales se anaden primero para que tengan
# siempre indices fijos: <PAD>=0
vocab.add_token('<PAD>')   # indice 0


# Recorremos todo el corpus añadiendo cada token unico
for sentence in sentences:
    for token in sentence:
        vocab.add_token(token)

print(f'Vocabulario construido.')
print(f'  Tamaño total : {vocab.size} tokens unicos')
print(f'  Indice <PAD>  : {vocab.token_to_idx["<PAD>"]}')
print(f'  Indice <EOL>  : {vocab.token_to_idx.get("<EOL>", "No encontrado")}')
print(f'  Indice "vida" : {vocab.token_to_idx.get("vida", "No encontrado")}')
print(f'\nEjemplo encode : {vocab.encode(["vida", "de", "mi"])}')
print(f'Ejemplo decode : {vocab.decode(vocab.encode(["vida", "de", "mi"]))}')

Vocabulario construido.
  Tamaño total : 14962 tokens unicos
  Indice <PAD>  : 0
  Indice <EOL>  : 7
  Indice "vida" : 321

Ejemplo encode : [321, 66, 5]
Ejemplo decode : ['vida', 'de', 'mi']


---
## PASO 3: Preparación del Dataset (Ventanas de Contexto)

### ¿Cómo aprende el modelo?
El modelo aprende a **predecir la siguiente palabra** dados los `context_size` tokens anteriores. Con `context_size=3` y la canción `['Ya', 'te', 'dije', 'frases', '<EOL>']`:

```
CONTEXTO (entrada)              OBJETIVO (a predecir)
['<PAD>', '<PAD>', '<PAD>']  ->  'Ya'
['<PAD>', '<PAD>', 'Ya']     ->  'te'
['<PAD>', 'Ya',   'te']      ->  'dije'
['Ya',    'te',   'dije']    ->  'frases'
['te',    'dije', 'frases']  ->  '<EOL>'
```

### ¿Por qué `<PAD>` al inicio?
Al principio de la canción no hay palabras previas. `<PAD>` actúa como **relleno neutro** para completar la ventana, permitiendo que el modelo también aprenda a predecir las primeras palabras de una canción.

### ¿Por qué los tensores deben tener tamaño fijo en PyTorch?
PyTorch agrupa los ejemplos en **batches** para aprovechar la GPU. Las multiplicaciones matriciales requieren que **todos los vectores del batch tengan exactamente el mismo tamaño**. El padding lo garantiza.

In [6]:
def prepare_bengio_dataset(sentences, vocab, context_size):
    """
    Genera todos los pares (contexto, objetivo) para entrenar el modelo.

    Para cada posicion i de cada cancion crea un ejemplo:
        Entrada  (contexto): los 'context_size' tokens ANTERIORES a i
        Salida   (objetivo): el token EN la posicion i

    Las posiciones iniciales se rellenan con <PAD> para poder generar
    ejemplos desde la primera palabra real de cada cancion.

    Args:
        sentences (list):    Lista de canciones tokenizadas.
        vocab (Vocabulary):  Vocabulario ya construido.
        context_size (int):  Tamaño de la ventana de contexto.

    Returns:
        list[tuple]: Lista de pares (lista_ids_contexto, id_objetivo).
    """
    data = []
    pad_idx = vocab.token_to_idx['<PAD>']

    for sentence in sentences:
        # Convertimos palabras a indices numericos
        encoded = vocab.encode(sentence)

        # Añadimos 'context_size' PADs al inicio
        # La primera palabra real tendra un contexto completo de PADs
        padded = [pad_idx] * context_size + encoded

        # Deslizamos la ventana de izquierda a derecha
        for i in range(len(padded) - context_size):
            context = padded[i : i + context_size]   # ventana de entrada
            target  = padded[i + context_size]        # token a predecir
            data.append((context, target))

    return data


CONTEXT_SIZE = 3
bengio_data  = prepare_bengio_dataset(sentences, vocab, context_size=CONTEXT_SIZE)

print(f'Dataset preparado.')
print(f'  Total de ejemplos: {len(bengio_data)}')
print(f'\nPrimeros 6 ejemplos (mostrando palabras, no IDs):')
print(f'{"CONTEXTO":45s}  ->  OBJETIVO')
print('-' * 65)
for ctx_ids, tgt_id in bengio_data[:6]:
    ctx_words = vocab.decode(ctx_ids)
    tgt_word  = vocab.idx_to_token[tgt_id]
    print(f'{str(ctx_words):45s}  ->  "{tgt_word}"')

Dataset preparado.
  Total de ejemplos: 278211

Primeros 6 ejemplos (mostrando palabras, no IDs):
CONTEXTO                                       ->  OBJETIVO
-----------------------------------------------------------------
['<PAD>', '<PAD>', '<PAD>']                    ->  "Tú"
['<PAD>', '<PAD>', 'Tú']                       ->  "y"
['<PAD>', 'Tú', 'y']                           ->  "yo"
['Tú', 'y', 'yo']                              ->  "en"
['y', 'yo', 'en']                              ->  "mi"
['yo', 'en', 'mi']                             ->  "habitación"


---
## PASO 4: Definición del Modelo (Red de Bengio)

### Arquitectura completa

```
ENTRADA:  [idx1, idx2, idx3]   <- context_size indices de palabras
              |
    +---------+---------+
    |   EMBEDDING       |  Cada indice -> vector de embed_size numeros
    |   nn.Embedding    |  Los 3 vectores se CONCATENAN en uno solo
    +---------+---------+  Forma resultante: [context_size * embed_size]
              |
    +---------+---------+
    |  CAPA OCULTA      |  Transformacion no lineal
    |  fc1 + tanh       |  Aprende patrones complejos del contexto
    +---------+---------+  Forma resultante: [hidden_dim]
              |
    +---------+---------+
    |  CAPA DE SALIDA   |  Una puntuacion (logit) por cada palabra
    |  fc2              |  Forma resultante: [vocab_size]
    +---------+---------+
              |
SALIDA:   [0.1, -2.3, 5.7, ...]  <- vocab_size logits
          La palabra con mayor logit es la más probable
```

### ¿Qué es un Embedding?
Un embedding es una representación vectorial densa de una palabra. En lugar de un vector one-hot (un 1 entre miles de ceros), el modelo aprende vectores compactos donde **palabras con significados parecidos quedan cerca en el espacio vectorial**.



In [7]:
class BengioNN(nn.Module):
    """
    Red Neuronal de Lenguaje de Bengio (2003).

    Dado un contexto de 'context_size' palabras, predice la siguiente.

    Arquitectura:
        Embedding -> Concatenacion -> Capa oculta (tanh) -> Capa de salida

    Args:
        vocab_size (int):   Numero de palabras en el vocabulario.
        context_size (int): Numero de palabras de contexto de entrada.
        embed_size (int):   Dimension de los vectores de embedding.
        hidden_dim (int):   Numero de neuronas en la capa oculta.
    """

    def __init__(self, vocab_size, context_size, embed_size, hidden_dim):
        super(BengioNN, self).__init__()

        self.context_size = context_size

        # Tabla de embeddings: vocab_size filas, cada una de embed_size dimensiones
        # Se inicializa aleatoriamente y se APRENDE durante el entrenamiento
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Capa oculta: recibe los context_size embeddings concatenados
        # Entrada: context_size * embed_size   Salida: hidden_dim
        self.fc1 = nn.Linear(context_size * embed_size, hidden_dim)

        # Capa de salida: un logit por cada palabra del vocabulario
        # Entrada: hidden_dim   Salida: vocab_size
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        """
        Propagacion hacia adelante (forward pass).

        Args:
            x: Tensor [batch_size, context_size] con indices de palabras.

        Returns:
            logits: Tensor [batch_size, vocab_size].
        """
        # 1. Lookup de embeddings
        #    [batch, context_size] -> [batch, context_size, embed_size]
        emb = self.embedding(x)

        # 2. Aplanamos los embeddings del contexto en un unico vector
        #    [batch, context_size, embed_size] -> [batch, context_size * embed_size]
        emb = emb.view(x.size(0), -1)

        # 3. Capa oculta con tanh (no-linealidad)
        #    [batch, context_size * embed_size] -> [batch, hidden_dim]
        hidden = F.tanh(self.fc1(emb))

        # 4. Capa de salida: logits
        #    [batch, hidden_dim] -> [batch, vocab_size]
        logits = self.fc2(hidden)

        return logits


# Hiperparametros del modelo
EMBED_SIZE = 128   # Dimension de cada vector de embedding
HIDDEN_DIM = 256   # Neuronas en la capa oculta

model = BengioNN(
    vocab_size   = vocab.size,
    embed_size   = EMBED_SIZE,
    hidden_dim   = HIDDEN_DIM,
    context_size = CONTEXT_SIZE
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Modelo creado.')
print(f'\nArquitectura:')
print(model)
print(f'\nParametros entrenables: {total_params:,}')

Modelo creado.

Arquitectura:
BengioNN(
  (embedding): Embedding(14962, 128)
  (fc1): Linear(in_features=384, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=14962, bias=True)
)

Parametros entrenables: 5,858,930


---
## PASO 5: Dataset y DataLoader de PyTorch

### ¿Qué es un Dataset de PyTorch?
PyTorch define una clase abstracta `Dataset` que debemos heredar. Solo necesita dos métodos:
- `__len__`: ¿Cuántos ejemplos hay en total?
- `__getitem__`: Dame el ejemplo número `idx` como tensores.

### ¿Qué es un DataLoader?
El `DataLoader` se encarga automáticamente de:
1. **Agrupar ejemplos en batches** para procesar varios a la vez.
2. **Mezclar aleatoriamente** los datos en cada época (`shuffle=True`), lo que mejora el entrenamiento evitando que el modelo memorice el orden.
3. **Paralelizar** la carga de datos si se usa GPU.

Procesar en batches es mucho más eficiente porque aprovecha la capacidad de las GPU para hacer miles de multiplicaciones matriciales en paralelo.

In [8]:
class CustomDataset(Dataset):
    """
    Envuelve la lista de pares (contexto, objetivo) en el
    formato que PyTorch necesita para usar DataLoader.
    """

    def __init__(self, data):
        self.data = data

    def __len__(self):
        """Numero total de ejemplos."""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Devuelve el ejemplo numero 'idx' como tensores PyTorch.

        Returns:
            inputs: Tensor Long [context_size] con los indices del contexto.
            target: Tensor Long [] (escalar) con el indice objetivo.
        """
        inputs, target = self.data[idx]
        # dtype=torch.long porque son indices enteros (requerido por nn.Embedding)
        return torch.tensor(inputs, dtype=torch.long), torch.tensor(target, dtype=torch.long)


BATCH_SIZE    = 32
dataset       = CustomDataset(bengio_data)
bengio_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Dataset y DataLoader listos.')
print(f'  Ejemplos totales : {len(dataset)}')
print(f'  Batch size       : {BATCH_SIZE}')
print(f'  Batches por epoca: {len(bengio_loader)}')

# Inspeccionamos las dimensiones de un batch
sample_inputs, sample_targets = next(iter(bengio_loader))
print(f'\nForma tensor entradas : {sample_inputs.shape}   (batch x context_size)')
print(f'Forma tensor objetivos: {sample_targets.shape}  (batch)')

Dataset y DataLoader listos.
  Ejemplos totales : 278211
  Batch size       : 32
  Batches por epoca: 8695

Forma tensor entradas : torch.Size([32, 3])   (batch x context_size)
Forma tensor objetivos: torch.Size([32])  (batch)


---
## PASO 6: Entrenamiento del Modelo

### El ciclo de entrenamiento

Para cada batch de datos, se repiten 4 pasos:

```
PASO 1 - FORWARD PASS
   El modelo recibe el contexto y produce logits (puntuaciones) para cada
   palabra del vocabulario.

PASO 2 - CALCULO DE LA PERDIDA (Loss)
   Se mide cuanto se equivoca el modelo usando Cross-Entropy.
   Si el modelo asigna alta probabilidad a la palabra correcta -> perdida baja.
   Si asigna baja probabilidad -> perdida alta.

PASO 3 - BACKWARD PASS (Backpropagation)
   Se calcula el gradiente: cuanto debe cambiar cada peso para reducir el error.

PASO 4 - ACTUALIZACION DE PESOS
   El optimizador Adam ajusta todos los pesos en la direccion del gradiente.
```

### Métrica: Perplejidad
La **perplejidad** es la métrica estándar para modelos de lenguaje:

```
Perplexity = e^(loss)
```

Una perplejidad de N significa que el modelo está tan confundido como si eligiera aleatoriamente entre N palabras. Cuanto más baja, mejor. Una perplejidad de 1 sería predicción perfecta.

In [9]:
def train_model(model, data_loader, vocab, num_epochs, learning_rate, device='cpu'):
    """
    Entrena el modelo de lenguaje de Bengio.

    Args:
        model:          Modelo BengioNN.
        data_loader:    DataLoader con los datos.
        vocab:          Vocabulario (necesitamos el indice de <PAD>).
        num_epochs:     Numero de vueltas completas al dataset.
        learning_rate:  Tasa de aprendizaje para Adam.
        device:         'cpu' o 'cuda'.
    """
    model.to(device)
    model.train()   # Modo entrenamiento

    # Cross-Entropy: mide el error de prediccion
    # ignore_index=<PAD>: los tokens de relleno no contribuyen al error
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.token_to_idx['<PAD>'])

    # Adam: ajusta la tasa de aprendizaje automaticamente para cada parametro
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f'Iniciando entrenamiento en {device}')
    print(f'  Epocas: {num_epochs}  |  LR: {learning_rate}  |  Batch: {data_loader.batch_size}')
    print('-' * 60)

    for epoch in range(num_epochs):
        total_loss = 0.0

        # Reemplaza la línea del for con esta:
        for inputs, targets in tqdm(data_loader, desc=f"Epoca {epoch+1}/{num_epochs}"):
            inputs  = inputs.to(device)
            targets = targets.to(device)

            # PASO 1: Forward pass
            outputs = model(inputs)          # [batch, vocab_size]

            # PASO 2: Calculo de la perdida
            loss = criterion(outputs, targets)

            # PASO 3: Backward pass
            optimizer.zero_grad()   # Limpiamos gradientes del paso anterior
            loss.backward()         # Calculamos gradientes

            # PASO 4: Actualizacion de pesos
            optimizer.step()

            total_loss += loss.item()

        avg_loss   = total_loss / len(data_loader)
        perplexity = torch.exp(torch.tensor(avg_loss)).item()

        print(f'Epoca [{epoch+1:3d}/{num_epochs}]  |  Loss: {avg_loss:.4f}  |  Perplejidad: {perplexity:.2f}')

    print('-' * 60)
    print('Entrenamiento completado.')


NUM_EPOCHS    = 10
LEARNING_RATE = 0.0001
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_model(
    model         = model,
    data_loader   = bengio_loader,
    vocab         = vocab,
    num_epochs    = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    device        = device
)

Iniciando entrenamiento en cuda
  Epocas: 10  |  LR: 0.0001  |  Batch: 32
------------------------------------------------------------


Epoca 1/10: 100%|██████████| 8695/8695 [00:26<00:00, 330.37it/s]


Epoca [  1/10]  |  Loss: 5.9818  |  Perplejidad: 396.14


Epoca 2/10: 100%|██████████| 8695/8695 [00:27<00:00, 321.32it/s]


Epoca [  2/10]  |  Loss: 4.9219  |  Perplejidad: 137.26


...

Epoca [  8/10]  |  Loss: 3.4784  |  Perplejidad: 32.41


Epoca 9/10: 100%|██████████| 8695/8695 [00:34<00:00, 250.26it/s]


Epoca [  9/10]  |  Loss: 3.3384  |  Perplejidad: 28.18


Epoca 10/10: 100%|██████████| 8695/8695 [00:22<00:00, 380.13it/s]

Epoca [ 10/10]  |  Loss: 3.2108  |  Perplejidad: 24.80
------------------------------------------------------------
Entrenamiento completado.


---
## PASO 7: Generación de Texto con Muestreo Top-K

### ¿Cómo genera texto el modelo?
Una vez entrenado, el modelo genera texto de forma **autoregresiva**: predice la siguiente palabra, la añade al contexto, y vuelve a predecir. Se repite hasta alcanzar la longitud máxima.

```
Semilla: ['Vida']

Paso 1:  Contexto ['<PAD>', '<PAD>', 'Vida']  ->  predice 'de'
Paso 2:  Contexto ['<PAD>', 'Vida',  'de'  ]  ->  predice 'mi'
Paso 3:  Contexto ['Vida',  'de',    'mi'  ]  ->  predice 'vida'
...
Resultado: 'Vida de mi vida <EOL> adonde hemos llegado'
```

### ¿Qué es el muestreo Top-K?
En lugar de elegir siempre la palabra **más probable** (greedy decoding, que produce textos repetitivos), el muestreo Top-K:
1. Selecciona las **K palabras más probables**.
2. Normaliza sus probabilidades con Softmax.
3. **Muestrea aleatoriamente** una de ellas, introduciendo variedad.

| K | Comportamiento |
|---|----------------|
| 1 | Greedy: determinista, puede ser repetitivo |
| 3-10 | Balance entre coherencia y variedad |
| 50+ | Muy creativo pero puede ser incoherente |

In [11]:
def generate_language_with_top_k_sampling(model, start_text, max_length, k, vocab, device='cpu'):
    """
    Genera texto usando el modelo entrenado con muestreo Top-K.

    El proceso es autoregresivo: cada token generado se añade al contexto
    para predecir el siguiente.

    Args:
        model:       Modelo BengioNN ya entrenado.
        start_text:  Lista de palabras semilla. Ej: ['La'].
        max_length:  Numero maximo de tokens a generar.
        k:           Candidatos para el muestreo Top-K.
        vocab:       Vocabulario.
        device:      'cpu' o 'cuda'.

    Returns:
        str: Texto generado (tokens separados por espacios).
    """
    input_tokens = vocab.encode(start_text)
    context_size = model.context_size
    pad_idx      = vocab.token_to_idx['<PAD>']

    model.to(device)
    model.eval()   # Modo evaluacion: desactiva comportamientos de entrenamiento

    with torch.no_grad():   # No calculamos gradientes: ahorra memoria y tiempo
        for _ in range(max_length):

            # Preparamos el contexto de tamaño fijo
            # Si hay menos tokens que context_size, rellenamos con PAD a la izquierda
            if len(input_tokens) < context_size:
                ctx = [pad_idx] * (context_size - len(input_tokens)) + input_tokens
            else:
                ctx = input_tokens[-context_size:]   # Ventana deslizante

            # Tensor de entrada: añade dimension de batch (tamaño 1)
            inputs = torch.tensor([ctx], device=device)   # [1, context_size]

            # Forward pass
            logits = model(inputs)[0]   # [vocab_size]

            # --- Muestreo Top-K ---
            # 1. Seleccionamos los K tokens con mayor puntuacion
            top_k_logits, top_k_indices = torch.topk(logits, k)

            # 2. Convertimos a probabilidades con Softmax
            probs = F.softmax(top_k_logits, dim=-1)

            # 3. Muestreamos uno aleatoriamente segun las probabilidades
            sampled_pos = torch.multinomial(probs, 1).item()
            next_token  = top_k_indices[sampled_pos].item()

            input_tokens.append(next_token)

            # Paramos si generamos un token de fin de secuencia
            if next_token == vocab.token_to_idx.get('<EOS>'):
                break

    return ' '.join(vocab.decode(input_tokens))


print('Texto generado por el modelo:')
print('=' * 60)

semillas = [['<PAD>'], ['<EOL>'], ['guitarra'], ['perro']]

for semilla in semillas:
    palabra = semilla[0]
    if palabra in vocab.token_to_idx:
        texto = generate_language_with_top_k_sampling(
            model      = model,
            start_text = semilla,
            max_length = 30,
            k          = 10,
            vocab      = vocab,
            device     = device
        )
        print(f'  [{palabra}] -> {texto}')
    else:
        print(f'  [{palabra}] -> Palabra no esta en el vocabulario')

Texto generado por el modelo:
  [<PAD>] -> <PAD> Ya no sé si me explico <EOL> Porque es que no se ve <EOL> Y que la luna , de esta noche te encontré <EOL> Pero es puro sentimiento <EOL>
  [<EOL>] -> <EOL> Es que te quiero decir <EOL> No quiero estar solo nunca más <EOL> Que yo quiero olvidar <EOL> Te digo que te puedo hacer <EOL> No me gusta decir ,
  [guitarra] -> guitarra , y en mi voz te lo debo a la boca <EOL> Y me iré , iré , iré , iré , iré <EOL> Y te llevaste el cenicero <EOL>
  [perro] -> perro que te ladre , princesa <EOL> Búscate en un millón de auroras <EOL> Y ninguna me enamora a poco <EOL> Vaya estando <EOL> En tus ojos <EOL> Y ahora vengo


---
## Cuestionario de Reflexion

---

**Pregunta 1:** En `read_and_tokenize`, el corpus mantiene las mayusculas originales (`Ya`, `Vida`, `La`...). Si convirtieras todo a minusculas antes de tokenizar, que ventaja tendria para el vocabulario? Que informacion se perderia?

**Tu respuesta:**
Pasar todo a minusculas reduce el tamano del vocabulario porque unifica variantes como `Vida`, `vida` y `VIDA` en un solo token. Eso mejora la cobertura y reduce dispersion de frecuencias. Como desventaja, se pierde informacion ortografica y semantica: nombres propios, inicios de frase, enfasis y posibles matices estilisticos.

---

**Pregunta 2:** El token `<EOL>` forma parte del vocabulario y el modelo aprende a predecirlo. Que ventaja tiene tratarlo como un token mas en lugar de eliminarlo durante la limpieza?

**Tu respuesta:**
Mantener `<EOL>` permite que el modelo aprenda la estructura de los versos y donde suelen terminar las lineas. Sin ese token, el texto se vuelve una secuencia plana y se pierde informacion de ritmo, pausas y segmentacion poetica, lo que suele empeorar la generacion de letras.

---

**Pregunta 3:** En la clase `Vocabulary` hemos añadido `<UNK>`. Aunque el corpus de entrenamiento no tiene palabras desconocidas, cuando seria imprescindible este token? Pon un ejemplo concreto.

**Tu respuesta:**
Es imprescindible en inferencia, cuando el usuario introduce palabras que no estaban en entrenamiento. Ejemplo: si la semilla es `['metaverso']` y ese termino no existe en el vocabulario, sin `<UNK>` fallaria el mapeo a indices; con `<UNK>` se representa de forma robusta y el modelo puede seguir generando.

---

**Pregunta 4:** La perplejidad baja con cada epoca. Que significaria una perplejidad final de 1.0? Seria eso deseable en un modelo que queremos usar para generar texto creativo?

**Tu respuesta:**
Una perplejidad de 1.0 implica prediccion perfecta del siguiente token (incertidumbre minima). En practica, en datos reales casi nunca ocurre y puede indicar sobreajuste si se logra solo en entrenamiento. Para generacion creativa no siempre es deseable: demasiada certeza suele producir texto menos diverso y mas repetitivo.

---

**Pregunta 5:** Compara el comportamiento del muestreo con `k=1` (greedy) frente a `k=10`. En que situaciones preferirías cada uno?

**Tu respuesta:**
Con `k=1` (greedy) la salida es determinista, mas estable y normalmente mas coherente localmente, util para evaluacion, depuracion o tareas donde importa precision. Con `k=10` hay aleatoriedad controlada: aumenta variedad y creatividad, util para generacion artistica o exploracion de alternativas, aunque con algo mas de riesgo de incoherencia.

---

**Pregunta 6 (Investigacion):** Si el vocabulario tuviera 100.000 palabras, la capa `fc2` y el Softmax serian enormemente costosos. Que tecnicas existen en PLN para reducir este coste computacional?

**Tu respuesta:**
Tecnicas habituales:
- **Hierarchical Softmax**: reduce el coste usando una estructura en arbol.
- **Sampled Softmax / Negative Sampling / NCE**: aproximan la normalizacion con subconjuntos de clases.
- **Adaptive Softmax**: asigna mas capacidad a palabras frecuentes y abarata las raras.
- **Subword tokenization** (BPE, WordPiece, Unigram): reduce vocabulario efectivo al trabajar con subpalabras.
- **Weight tying** entre embeddings de entrada/salida: reduce parametros y memoria.
- **Shortlists o vocabulario truncado por dominio/frecuencia** en escenarios controlados.